In [1]:
import pandas as pd

In [2]:
import os
os.getcwd()

'C:\\Users\\user\\deepthought'

In [7]:
df = pd.read_csv("D:\\deepthough\\node.tsv", sep='\t')

In [8]:
df

,id,parentId,type,text,options,target,signal
0,START,NaN,start,"Good evening. Before we close the day, let’s t...",NaN,NaN,NaN
1,A1_OPEN,START,question,How did today feel overall?,Smooth and productive|Mixed with ups and downs...,NaN,NaN
2,A1_D1,A1_OPEN,decision,NaN,answer=Smooth and productive|Mixed with ups an...,NaN,NaN
3,A1_Q_HIGH,A1_D1,question,"When things went well, what made the difference?",My preparation|Decisions I made in the moment|...,NaN,axis1:internal
4,A1_Q_LOW,A1_D1,question,"When things didn’t go as planned, what was you...",Focus on what I could still control|Wait for d...,NaN,axis1:external
5,A1_D2,A1_Q_HIGH,decision,NaN,answer=My preparation|Decisions I made in the ...,NaN,NaN
6,A1_D3,A1_Q_LOW,decision,NaN,answer=Focus on what I could still control:A1_...,NaN,NaN
7,A1_R_INT,A1_D2,reflection,You noticed where your actions shaped the day....,NaN,NaN,axis1:internal
8,A1_R_MIX,A1_D2,reflection,"Some things worked because of you, some becaus...",NaN,NaN,axis1:neutral
9,A1_R_EXT,A1_D3,reflection,Today may have felt driven by circumstances. S...,NaN,NaN,axis1:external


In [9]:
tree = df.set_index("id").to_dict(orient="index")

In [10]:
print(tree["START"])

{'parentId': nan, 'type': 'start', 'text': 'Good evening. Before we close the day, let’s take a few minutes to reflect.', 'options': nan, 'target': nan, 'signal': nan}


In [11]:
tree["A1_OPEN"]

{'parentId': 'START',
 'type': 'question',
 'text': 'How did today feel overall?',
 'options': 'Smooth and productive|Mixed with ups and downs|Mostly challenging|Draining and frustrating',
 'target': nan,
 'signal': nan}

In [13]:
df.fillna("")

,id,parentId,type,text,options,target,signal
0,START,,start,"Good evening. Before we close the day, let’s t...",,,
1,A1_OPEN,START,question,How did today feel overall?,Smooth and productive|Mixed with ups and downs...,,
2,A1_D1,A1_OPEN,decision,,answer=Smooth and productive|Mixed with ups an...,,
3,A1_Q_HIGH,A1_D1,question,"When things went well, what made the difference?",My preparation|Decisions I made in the moment|...,,axis1:internal
4,A1_Q_LOW,A1_D1,question,"When things didn’t go as planned, what was you...",Focus on what I could still control|Wait for d...,,axis1:external
5,A1_D2,A1_Q_HIGH,decision,,answer=My preparation|Decisions I made in the ...,,
6,A1_D3,A1_Q_LOW,decision,,answer=Focus on what I could still control:A1_...,,
7,A1_R_INT,A1_D2,reflection,You noticed where your actions shaped the day....,,,axis1:internal
8,A1_R_MIX,A1_D2,reflection,"Some things worked because of you, some becaus...",,,axis1:neutral
9,A1_R_EXT,A1_D3,reflection,Today may have felt driven by circumstances. S...,,,axis1:external


In [14]:
def get_children(node_id):
    return [nid for nid, node in tree.items() if node["parentId"] == node_id]

In [15]:
def parse_options(options_str):
    return options_str.split("|") if options_str else []

In [16]:
def parse_decision(decision_str):
    rules = []
    parts = decision_str.split(";")

    for part in parts:
        cond, target = part.split(":")
        values = cond.replace("answer=", "").split("|")
        rules.append((values, target))

    return rules

In [17]:
state = {
    "answers": {},
    "axis1": {"internal": 0, "external": 0, "neutral": 0},
    "axis2": {"contribution": 0, "entitlement": 0, "neutral": 0},
    "axis3": {"self": 0, "other": 0}
}

In [18]:
def update_signal(signal):
    # Check if signal is valid string
    if not isinstance(signal, str) or signal.strip() == "":
        return

    axis, value = signal.split(":")
    state[axis][value] += 1

In [19]:
def get_dominant(axis_dict):
    return max(axis_dict, key=axis_dict.get)

In [20]:
def interpolate(text):
    # Ensure text is string
    if not isinstance(text, str):
        text = ""

    # Replace answer placeholders
    for key, value in state["answers"].items():
        placeholder = "{" + key + ".answer}"
        text = text.replace(placeholder, value)

    # Replace axis summaries
    text = text.replace("{axis1.dominant}", get_dominant(state["axis1"]))
    text = text.replace("{axis2.dominant}", get_dominant(state["axis2"]))
    text = text.replace("{axis3.dominant}", get_dominant(state["axis3"]))

    return text

In [21]:
def run_tree():
    current = "START"
    prev_answer = None

    while True:
        node = tree[current]
        node_type = node["type"]
        text = interpolate(node["text"])

        # Display text
        if text:
            print("\n" + text)

        # Update signal
        update_signal(node["signal"])

        # Handle node types
        if node_type == "start":
            children = get_children(current)
            current = children[0]

        elif node_type == "question":
            options = parse_options(node["options"])

            for i, opt in enumerate(options, 1):
                print(f"{i}. {opt}")

            while True:
                try:
                    choice = int(input("Select option: ")) - 1
                    if 0 <= choice < len(options):
                        break
                    else:
                        print("Invalid option. Please choose a number from the list.")
                except ValueError:
                    print("Invalid input. Please enter a number.")

            answer = options[choice]

            state["answers"][current] = answer
            prev_answer = answer

            children = get_children(current)
            current = children[0]

        elif node_type == "decision":
            rules = parse_decision(node["options"])

            for values, target in rules:
                if prev_answer in values:
                    current = target
                    break

        elif node_type in ["reflection", "bridge"]:
            input("\nPress Enter to continue...")

            # Use target if available
            if pd.notna(node["target"]):  # Check if target is not NaN
                current = node["target"]
            else:
                children = get_children(current)
                if not children:
                    print("Error: No next node found.")
                    break
                current = children[0]

        elif node_type == "summary":
            print("\n--- SUMMARY ---")

            if pd.notna(node["target"]):  # Check if target is not NaN
                current = node["target"]
            else:
                children = get_children(current)
                if not children:
                    print("Error: No next node found.")
                    break
                current = children[0]

        elif node_type == "end":
            print("\nSession complete.")
            break

In [ ]:
run_tree()


Good evening. Before we close the day, let’s take a few minutes to reflect.

How did today feel overall?
1. Smooth and productive
2. Mixed with ups and downs
3. Mostly challenging
4. Draining and frustrating


Select option:  1



When things went well, what made the difference?
1. My preparation
2. Decisions I made in the moment
3. Support from others
4. Things just worked out


Select option:  2



You noticed where your actions shaped the day. Even small decisions reflect agency.


...


You contributed beyond what was required. That kind of effort strengthens teams over time.



Press Enter to continue... 



Now let’s widen the lens a bit further.



Press Enter to continue... 



When you reflect on today’s most important moment, who comes to mind first?
1. Just myself
2. My immediate team
3. A specific colleague
4. The customer or end user
